

## Correspondance avant tout prétraitement (anti-fuite / anti-leakage)
Point clé : **on fait d’abord la correspondance ECG↔Respiration**, et *ensuite seulement* on calcule les caractéristiques (features).
- On construit une table de paires (`pairs_raw`) en associant un fichier ECG et un fichier respiration appartenant au **même sujet** et à la **même session/date**.
- Chaque paire reçoit un identifiant stable `pair_id` (ex. `healthy001_<session>` ou `diabetes012_<session>`).

## Séparation train/test par sujet
Pour éviter toute fuite d’information, on sépare **par sujet** (`GroupShuffleSplit`) :
- Tous les enregistrements d’un même sujet vont **soit** en train **soit** en test.
- On extrait ensuite les features et on entraîne/évalue les modèles sur cette base.

## Conventions de parsing
- `subject_id` est extrait du chemin (ex. `001`, `012`).
- Cas spécial : dossier `012_diabetes` (normalisé en `012` et étiqueté diabétique).

In [ ]:
from pathlib import Path
import re
import os

import numpy as np
import pandas as pd

import scipy.signal as sp_signal
from scipy.signal import find_peaks, welch

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GroupShuffleSplit

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    confusion_matrix, classification_report,
 )


In [ ]:
from pathlib import Path
import re
import os

# ===============================
# CONFIG (Kaggle)
# ===============================
BREATH_HEALTHY_ROOT = Path('/data/data/healthy_subset_sensor_data/healthy_subset_sensor_data')
BREATH_DIAB_ROOT    = Path('/data/data/diabetes_subset_sensor_data/diabetes_subset_sensor_data')

MIN_ROWS = 64_000

PERSON_NUM_RE = re.compile(r'^(\d{1,3})')  # '001' ou '012_diabetes' -> 012

def is_valid_length(csv_file: Path, min_rows: int = MIN_ROWS) -> bool:
    """Vérifie si un CSV contient au moins min_rows lignes (streaming)."""
    try:
        with open(csv_file, 'r', encoding='utf-8', errors='ignore') as f:
            for i, _ in enumerate(f, 1):
                if i >= min_rows:
                    return True
        return False
    except Exception:
        return False

def subject_id_from_path(file_path: Path) -> str:
    """Extrait le sujet (001) depuis .../<person>/sensor_data/... même si <person> = '012_diabetes'."""
    parts = file_path.as_posix().split('/')
    if 'sensor_data' in parts:
        person_folder = parts[parts.index('sensor_data') - 1]
    else:
        person_folder = file_path.parent.name
    m = PERSON_NUM_RE.match(person_folder)
    if not m:
        return 'Unknown'
    return f"{int(m.group(1)):03d}"

def session_id_from_path(file_path: Path) -> str:
    """Extrait l'identifiant de session/date depuis .../sensor_data/<SESSION>/..."""
    parts = file_path.as_posix().split('/')
    if 'sensor_data' in parts:
        idx = parts.index('sensor_data')
        if idx + 1 < len(parts):
            return parts[idx + 1]
    return file_path.parent.name

def label_from_person_folder(person_folder: str, default_y: int) -> int:
    """Override: si le dossier contient 'diabetes' => 1, sinon default_y."""
    if 'diabetes' in person_folder.lower():
        return 1
    return int(default_y)

def collect_breathing_sessions(root: Path, default_y: int) -> pd.DataFrame:
    rows = []
    if not root.exists():
        return pd.DataFrame(columns=['file_path', 'subject_id', 'session_id', 'y'])

    for person_dir in root.iterdir():
        if not person_dir.is_dir():
            continue
        # inclut '012_diabetes' (pas uniquement .isdigit())
        if PERSON_NUM_RE.match(person_dir.name) is None:
            continue
        sensor_root = person_dir / 'sensor_data'
        if not sensor_root.exists():
            continue
        y = label_from_person_folder(person_dir.name, default_y)
        for csv_file in sensor_root.rglob('*.csv'):
            if 'breathing' not in csv_file.name.lower():
                continue
            if not is_valid_length(csv_file, MIN_ROWS):
                continue
            rows.append({
                'file_path': str(csv_file),
                'subject_id': subject_id_from_path(csv_file),
                'session_id': session_id_from_path(csv_file),
                'y': int(y),
            })
    return pd.DataFrame(rows)

breath_sessions = pd.concat([
    collect_breathing_sessions(BREATH_HEALTHY_ROOT, default_y=0),
    collect_breathing_sessions(BREATH_DIAB_ROOT,    default_y=1),
], ignore_index=True)

print('Breathing sessions:', len(breath_sessions))
print('Breathing subjects:', breath_sessions['subject_id'].nunique() if len(breath_sessions) else 0)
print('Breathing sessions(unique):', breath_sessions[['subject_id','session_id']].drop_duplicates().shape[0] if len(breath_sessions) else 0)
print('Breathing label counts:', breath_sessions['y'].value_counts(dropna=False).to_dict() if len(breath_sessions) else {})
display(breath_sessions.head())

Breathing sessions: 126
Breathing subjects: 20
Breathing sessions(unique): 126
Breathing label counts: {0: 79, 1: 47}


,file_path,subject_id,session_id,y
0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,018,2014_10_03-06_45_21,0
1,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,018,2014_10_04-08_10_52,0
2,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,018,2014_10_01-08_00_48,0
3,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,018,2014_10_02-07_36_31,0
4,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,016,2014_10_01-12_07_54,0


In [ ]:
d =["data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/001/sensor_data/2014_10_01-10_09_39/2014_10_01-10_09_39_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/001/sensor_data/2014_10_02-10_56_44/2014_10_02-10_56_44_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/001/sensor_data/2014_10_03-06_36_24/2014_10_03-06_36_24_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/001/sensor_data/2014_10_04-06_34_57/2014_10_04-06_34_57_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/002/sensor_data/2014_10_01-12_35_54/2014_10_01-12_35_54_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/002/sensor_data/2014_10_01-20_29_57/2014_10_01-20_29_57_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/002/sensor_data/2014_10_02-06_44_21/2014_10_02-06_44_21_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/002/sensor_data/2014_10_03-06_33_20/2014_10_03-06_33_20_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/002/sensor_data/2014_10_03-14_06_08/2014_10_03-14_06_08_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/002/sensor_data/2014_10_04-07_01_03/2014_10_04-07_01_03_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/002/sensor_data/2014_10_04-17_43_12/2014_10_04-17_43_12_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/003/sensor_data/2014_10_01-06_00_57/2014_10_01-06_00_57_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/003/sensor_data/2014_10_02-09_24_32/2014_10_02-09_24_32_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/003/sensor_data/2014_10_03-00_54_40/2014_10_03-00_54_40_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/003/sensor_data/2014_10_03-18_06_15/2014_10_03-18_06_15_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/003/sensor_data/2014_10_04-05_33_24/2014_10_04-05_33_24_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/004/sensor_data/2014_10_01-06_55_10/2014_10_01-06_55_10_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/004/sensor_data/2014_10_01-19_33_00/2014_10_01-19_33_00_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/004/sensor_data/2014_10_02-07_21_06/2014_10_02-07_21_06_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/004/sensor_data/2014_10_03-08_05_42/2014_10_03-08_05_42_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/004/sensor_data/2014_10_04-06_32_58/2014_10_04-06_32_58_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/004/sensor_data/2014_10_04-16_49_30/2014_10_04-16_49_30_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/005/sensor_data/2014_10_01-10_27_42/2014_10_01-10_27_42_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/005/sensor_data/2014_10_01-10_40_51/2014_10_01-10_40_51_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/005/sensor_data/2014_10_02-09_23_58/2014_10_02-09_23_58_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/005/sensor_data/2014_10_02-11_40_30/2014_10_02-11_40_30_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/005/sensor_data/2014_10_03-08_42_25/2014_10_03-08_42_25_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/005/sensor_data/2014_10_04-08_02_17/2014_10_04-08_02_17_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/006/sensor_data/2014_10_01-06_46_44/2014_10_01-06_46_44_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/006/sensor_data/2014_10_01-15_08_04/2014_10_01-15_08_04_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/006/sensor_data/2014_10_02-08_03_53/2014_10_02-08_03_53_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/006/sensor_data/2014_10_02-15_13_03/2014_10_02-15_13_03_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/006/sensor_data/2014_10_03-08_57_56/2014_10_03-08_57_56_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/006/sensor_data/2014_10_04-10_30_26/2014_10_04-10_30_26_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/007/sensor_data/2014_10_01-08_42_43/2014_10_01-08_42_43_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/007/sensor_data/2014_10_02-07_52_44/2014_10_02-07_52_44_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/007/sensor_data/2014_10_03-06_46_57/2014_10_03-06_46_57_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/007/sensor_data/2014_10_04-08_27_21/2014_10_04-08_27_21_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/008/sensor_data/2014_10_01-06_43_00/2014_10_01-06_43_00_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/008/sensor_data/2014_10_02-10_13_52/2014_10_02-10_13_52_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/008/sensor_data/2014_10_03-10_51_39/2014_10_03-10_51_39_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/008/sensor_data/2014_10_03-22_54_51/2014_10_03-22_54_51_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/009/sensor_data/2014_10_01-05_59_30/2014_10_01-05_59_30_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/009/sensor_data/2014_10_02-06_14_52/2014_10_02-06_14_52_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/009/sensor_data/2014_10_03-08_21_59/2014_10_03-08_21_59_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/009/sensor_data/2014_10_04-09_09_29/2014_10_04-09_09_29_ECG.csv",
"data/data/diabetes_subset_ecg_data/diabetes_subset_ecg_data/009/sensor_data/2014_10_04-15_03_37/2014_10_04-15_03_37_ECG.csv"]
h =["data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/001/sensor_data/2014_10_01-12_50_01/2014_10_01-12_50_01_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/001/sensor_data/2014_10_02-07_11_17/2014_10_02-07_11_17_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/001/sensor_data/2014_10_03-07_50_09/2014_10_03-07_50_09_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/001/sensor_data/2014_10_04-07_31_03/2014_10_04-07_31_03_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/002/sensor_data/2014_10_01-06_17_49/2014_10_01-06_17_49_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/002/sensor_data/2014_10_02-06_27_58/2014_10_02-06_27_58_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/002/sensor_data/2014_10_03-06_30_00/2014_10_03-06_30_00_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/002/sensor_data/2014_10_04-06_24_41/2014_10_04-06_24_41_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/003/sensor_data/2014_10_01-10_14_30/2014_10_01-10_14_30_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/003/sensor_data/2014_10_02-08_37_05/2014_10_02-08_37_05_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/003/sensor_data/2014_10_03-08_37_23/2014_10_03-08_37_23_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/003/sensor_data/2014_10_04-08_47_14/2014_10_04-08_47_14_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/004/sensor_data/2014_10_01-06_25_47/2014_10_01-06_25_47_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/004/sensor_data/2014_10_02-06_22_16/2014_10_02-06_22_16_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/004/sensor_data/2014_10_02-15_58_32/2014_10_02-15_58_32_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/004/sensor_data/2014_10_03-06_20_53/2014_10_03-06_20_53_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/004/sensor_data/2014_10_04-06_17_01/2014_10_04-06_17_01_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/005/sensor_data/2014_10_01-10_29_04/2014_10_01-10_29_04_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/005/sensor_data/2014_10_02-08_10_52/2014_10_02-08_10_52_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/005/sensor_data/2014_10_03-08_01_29/2014_10_03-08_01_29_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/005/sensor_data/2014_10_04-08_10_40/2014_10_04-08_10_40_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/006/sensor_data/2014_10_01-09_04_15/2014_10_01-09_04_15_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/006/sensor_data/2014_10_02-06_50_16/2014_10_02-06_50_16_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/006/sensor_data/2014_10_03-07_48_25/2014_10_03-07_48_25_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/006/sensor_data/2014_10_04-08_25_08/2014_10_04-08_25_08_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/007/sensor_data/2014_10_02-06_32_36/2014_10_02-06_32_36_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/007/sensor_data/2014_10_02-13_10_03/2014_10_02-13_10_03_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/007/sensor_data/2014_10_03-06_43_59/2014_10_03-06_43_59_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/007/sensor_data/2014_10_03-21_18_12/2014_10_03-21_18_12_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/007/sensor_data/2014_10_03-21_45_56/2014_10_03-21_45_56_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/007/sensor_data/2014_10_04-08_08_25/2014_10_04-08_08_25_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/007/sensor_data/2014_10_04-21_44_52/2014_10_04-21_44_52_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/007/sensor_data/2014_10_05-10_23_50/2014_10_05-10_23_50_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/008/sensor_data/2014_10_01-10_04_02/2014_10_01-10_04_02_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/008/sensor_data/2014_10_02-09_02_32/2014_10_02-09_02_32_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/008/sensor_data/2014_10_03-09_12_30/2014_10_03-09_12_30_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/008/sensor_data/2014_10_04-09_19_55/2014_10_04-09_19_55_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/009/sensor_data/2014_10_01-07_30_11/2014_10_01-07_30_11_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/009/sensor_data/2014_10_02-07_05_46/2014_10_02-07_05_46_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/009/sensor_data/2014_10_03-07_31_26/2014_10_03-07_31_26_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/009/sensor_data/2014_10_04-06_09_00/2014_10_04-06_09_00_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/010/sensor_data/2014_10_01-11_26_27/2014_10_01-11_26_27_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/010/sensor_data/2014_10_02-09_34_42/2014_10_02-09_34_42_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/010/sensor_data/2014_10_03-09_25_35/2014_10_03-09_25_35_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/010/sensor_data/2014_10_04-10_32_46/2014_10_04-10_32_46_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/011/sensor_data/2014_10_01-11_32_11/2014_10_01-11_32_11_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/011/sensor_data/2014_10_02-08_45_01/2014_10_02-08_45_01_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/011/sensor_data/2014_10_03-07_37_32/2014_10_03-07_37_32_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/011/sensor_data/2014_10_04-07_54_56/2014_10_04-07_54_56_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/012_diabetes/sensor_data/2014_10_01-06_20_34/2014_10_01-06_20_34_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/012_diabetes/sensor_data/2014_10_02-06_11_43/2014_10_02-06_11_43_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/012_diabetes/sensor_data/2014_10_03-06_06_15/2014_10_03-06_06_15_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/013/sensor_data/2014_10_01-11_02_54/2014_10_01-11_02_54_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/013/sensor_data/2014_10_02-08_13_56/2014_10_02-08_13_56_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/013/sensor_data/2014_10_03-09_31_23/2014_10_03-09_31_23_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/013/sensor_data/2014_10_04-09_17_41/2014_10_04-09_17_41_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/014/sensor_data/2014_10_01-09_51_57/2014_10_01-09_51_57_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/014/sensor_data/2014_10_02-07_06_16/2014_10_02-07_06_16_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/014/sensor_data/2014_10_03-07_07_52/2014_10_03-07_07_52_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/014/sensor_data/2014_10_04-07_15_54/2014_10_04-07_15_54_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/015/sensor_data/2014_10_01-09_52_17/2014_10_01-09_52_17_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/015/sensor_data/2014_10_02-08_06_54/2014_10_02-08_06_54_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/015/sensor_data/2014_10_03-08_32_15/2014_10_03-08_32_15_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/015/sensor_data/2014_10_04-08_11_48/2014_10_04-08_11_48_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/016/sensor_data/2014_10_01-12_07_54/2014_10_01-12_07_54_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/016/sensor_data/2014_10_02-08_35_34/2014_10_02-08_35_34_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/016/sensor_data/2014_10_03-08_27_26/2014_10_03-08_27_26_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/016/sensor_data/2014_10_04-08_17_33/2014_10_04-08_17_33_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/017/sensor_data/2014_10_01-08_13_07/2014_10_01-08_13_07_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/017/sensor_data/2014_10_02-08_17_26/2014_10_02-08_17_26_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/017/sensor_data/2014_10_03-08_12_11/2014_10_03-08_12_11_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/017/sensor_data/2014_10_04-08_23_04/2014_10_04-08_23_04_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/018/sensor_data/2014_10_01-08_00_48/2014_10_01-08_00_48_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/018/sensor_data/2014_10_02-07_36_31/2014_10_02-07_36_31_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/018/sensor_data/2014_10_03-06_45_21/2014_10_03-06_45_21_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/018/sensor_data/2014_10_04-08_10_52/2014_10_04-08_10_52_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/019/sensor_data/2014_10_01-07_21_15/2014_10_01-07_21_15_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/019/sensor_data/2014_10_02-06_20_25/2014_10_02-06_20_25_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/019/sensor_data/2014_10_03-08_55_44/2014_10_03-08_55_44_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/019/sensor_data/2014_10_04-11_09_00/2014_10_04-11_09_00_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/020/sensor_data/2014_10_01-10_04_56/2014_10_01-10_04_56_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/020/sensor_data/2014_10_02-06_30_01/2014_10_02-06_30_01_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/020/sensor_data/2014_10_03-06_35_51/2014_10_03-06_35_51_ECG.csv",
"data/data/healthy_subset_ecg_data/healthy_subset_ecg_data/020/sensor_data/2014_10_04-06_08_25/2014_10_04-06_08_25_ECG.csv"]

In [10]:
# ===============================
# CORRESPONDANCE (avant tout processing)
# 1) Table ECG sessions (sujet, session/date, y)
# 2) Table Breathing sessions (déjà construite)
# 3) Jointure -> une paire (ECG, Breathing) par (sujet, session/date)
# ===============================

def ecg_sessions_from_lists(h_list, d_list) -> pd.DataFrame:
    rows = []
    for p in h_list:
        p = Path(p)
        parts = p.as_posix().split('/')
        person_folder = parts[parts.index('sensor_data') - 1] if 'sensor_data' in parts else p.parent.name
        y = label_from_person_folder(person_folder, default_y=0)
        rows.append({
            'file_path': str(p),
            'subject_id': subject_id_from_path(p),
            'session_id': session_id_from_path(p),
            'y': int(y),
        })
    for p in d_list:
        p = Path(p)
        parts = p.as_posix().split('/')
        person_folder = parts[parts.index('sensor_data') - 1] if 'sensor_data' in parts else p.parent.name
        y = label_from_person_folder(person_folder, default_y=1)
        rows.append({
            'file_path': str(p),
            'subject_id': subject_id_from_path(p),
            'session_id': session_id_from_path(p),
            'y': int(y),
        })
    df = pd.DataFrame(rows)
    df = df[df['subject_id'].ne('Unknown')].copy()
    return df

ecg_sessions = ecg_sessions_from_lists(h, d)
print('ECG sessions:', len(ecg_sessions))
print('ECG subjects:', ecg_sessions['subject_id'].nunique() if len(ecg_sessions) else 0)
print('ECG sessions(unique):', ecg_sessions[['subject_id','session_id']].drop_duplicates().shape[0] if len(ecg_sessions) else 0)
print('ECG label counts:', ecg_sessions['y'].value_counts(dropna=False).to_dict() if len(ecg_sessions) else {})
display(ecg_sessions.head())

# --- build pairs by exact (subject_id, session_id) match
pairs_raw = ecg_sessions.merge(
    breath_sessions,
    on=['subject_id', 'session_id'],
    how='inner',
    suffixes=('_ecg', '_breath'),
)

if len(pairs_raw):
    pairs_raw['y_pair'] = pairs_raw[['y_ecg', 'y_breath']].max(axis=1).astype(int)
    pairs_raw['prefix'] = np.where(pairs_raw['y_pair'].values == 1, 'diabetes', 'healthy')
    pairs_raw['pair_id'] = pairs_raw['prefix'] + pairs_raw['subject_id'] + '_' + pairs_raw['session_id']
    # Sanity: conflits de labels entre ECG et Breathing pour une même paire
    conflicts = pairs_raw[pairs_raw['y_ecg'].astype(int) != pairs_raw['y_breath'].astype(int)]
    print('Paired sessions:', len(pairs_raw))
    print('Paired subjects:', pairs_raw['subject_id'].nunique())
    print('Label conflicts (y_ecg != y_breath):', len(conflicts))
else:
    conflicts = pd.DataFrame()
    print('Paired sessions: 0  (aucune correspondance trouvée)')

# --- Unmatched diagnostic
ecg_keys = ecg_sessions[['subject_id','session_id']].drop_duplicates()
breath_keys = breath_sessions[['subject_id','session_id']].drop_duplicates()
unmatched_ecg = ecg_keys.merge(breath_keys, on=['subject_id','session_id'], how='left', indicator=True)
unmatched_ecg = unmatched_ecg[unmatched_ecg['_merge'].eq('left_only')].drop(columns=['_merge'])
unmatched_breath = breath_keys.merge(ecg_keys, on=['subject_id','session_id'], how='left', indicator=True)
unmatched_breath = unmatched_breath[unmatched_breath['_merge'].eq('left_only')].drop(columns=['_merge'])

print('Unmatched ECG sessions:', len(unmatched_ecg))
print('Unmatched Breathing sessions:', len(unmatched_breath))
display(pairs_raw[['pair_id','subject_id','session_id','file_path_ecg','file_path_breath','y_pair']].head() if len(pairs_raw) else pairs_raw)

ECG sessions: 131
ECG subjects: 20
ECG sessions(unique): 131
ECG label counts: {0: 81, 1: 50}


,file_path,subject_id,session_id,y
0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,001,2014_10_01-12_50_01,0
1,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,001,2014_10_02-07_11_17,0
2,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,001,2014_10_03-07_50_09,0
3,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,001,2014_10_04-07_31_03,0
4,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,002,2014_10_01-06_17_49,0


Paired sessions: 126
Paired subjects: 20
Label conflicts (y_ecg != y_breath): 0
Unmatched ECG sessions: 5
Unmatched Breathing sessions: 0


,pair_id,subject_id,session_id,file_path_ecg,file_path_breath,y_pair
0,healthy001_2014_10_01-12_50_01,001,2014_10_01-12_50_01,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,0
1,healthy001_2014_10_02-07_11_17,001,2014_10_02-07_11_17,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,0
2,healthy001_2014_10_03-07_50_09,001,2014_10_03-07_50_09,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,0
3,healthy001_2014_10_04-07_31_03,001,2014_10_04-07_31_03,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,0
4,healthy002_2014_10_01-06_17_49,002,2014_10_01-06_17_49,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,0


In [11]:
# Split EXACTEMENT comme dans tutoré ECG.ipynb (GroupShuffleSplit sujet-aware)
RANDOM_STATE = 42
TEST_SIZE = 0.2
print('Split config:', {'test_size': TEST_SIZE, 'random_state': RANDOM_STATE})

if 'pairs_raw' not in globals() or len(pairs_raw) == 0:
    raise ValueError('pairs_raw est vide: aucune paire ECG↔Breathing. Vérifie les paths et le parsing session_id.')

# Une étiquette par sujet (max) — et split par sujet
subject_table = (
    pairs_raw[['subject_id', 'y_pair']]
    .drop_duplicates()
    .groupby('subject_id', as_index=False)['y_pair'].max()
    .rename(columns={'y_pair': 'y'})
)

gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(subject_table['subject_id'], subject_table['y'], groups=subject_table['subject_id']))
train_subjects = set(subject_table.iloc[train_idx]['subject_id'].tolist())
test_subjects  = set(subject_table.iloc[test_idx]['subject_id'].tolist())

pairs_train = pairs_raw[pairs_raw['subject_id'].isin(train_subjects)].copy()
pairs_test  = pairs_raw[pairs_raw['subject_id'].isin(test_subjects)].copy()

print('Subjects train/test:', len(train_subjects), '/', len(test_subjects))
print('Pairs train/test:', len(pairs_train), '/', len(pairs_test))
print('Overlap subjects?', len(train_subjects.intersection(test_subjects)) > 0)
print('Train label counts:', pairs_train['y_pair'].value_counts().to_dict())
print('Test label counts:', pairs_test['y_pair'].value_counts().to_dict())

# Affiche les paires (id = healthy001_<date> / diabetes012_<date>)
cols_show = ['pair_id','subject_id','session_id','y_pair','file_path_ecg','file_path_breath']
display(pairs_train[cols_show].head(10))
display(pairs_test[cols_show].head(10))

Split config: {'test_size': 0.2, 'random_state': 42}
Subjects train/test: 16 / 4
Pairs train/test: 100 / 26
Overlap subjects? False
Train label counts: {0: 63, 1: 37}
Test label counts: {0: 16, 1: 10}


,pair_id,subject_id,session_id,y_pair,file_path_ecg,file_path_breath
8,healthy003_2014_10_01-10_14_30,003,2014_10_01-10_14_30,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
9,healthy003_2014_10_02-08_37_05,003,2014_10_02-08_37_05,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
10,healthy003_2014_10_03-08_37_23,003,2014_10_03-08_37_23,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
11,healthy003_2014_10_04-08_47_14,003,2014_10_04-08_47_14,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
12,healthy004_2014_10_01-06_25_47,004,2014_10_01-06_25_47,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
13,healthy004_2014_10_02-06_22_16,004,2014_10_02-06_22_16,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
14,healthy004_2014_10_02-15_58_32,004,2014_10_02-15_58_32,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
15,healthy004_2014_10_03-06_20_53,004,2014_10_03-06_20_53,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
16,healthy004_2014_10_04-06_17_01,004,2014_10_04-06_17_01,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
17,healthy005_2014_10_01-10_29_04,005,2014_10_01-10_29_04,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...


,pair_id,subject_id,session_id,y_pair,file_path_ecg,file_path_breath
0,healthy001_2014_10_01-12_50_01,001,2014_10_01-12_50_01,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
1,healthy001_2014_10_02-07_11_17,001,2014_10_02-07_11_17,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
2,healthy001_2014_10_03-07_50_09,001,2014_10_03-07_50_09,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
3,healthy001_2014_10_04-07_31_03,001,2014_10_04-07_31_03,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
4,healthy002_2014_10_01-06_17_49,002,2014_10_01-06_17_49,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
5,healthy002_2014_10_02-06_27_58,002,2014_10_02-06_27_58,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
6,healthy002_2014_10_03-06_30_00,002,2014_10_03-06_30_00,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
7,healthy002_2014_10_04-06_24_41,002,2014_10_04-06_24_41,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
62,healthy016_2014_10_01-12_07_54,016,2014_10_01-12_07_54,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
63,healthy016_2014_10_02-08_35_34,016,2014_10_02-08_35_34,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...,/kaggle/input/d1namo-ecg-glucose-data/healthy_...


## ECG features 




In [12]:
# ===============================
# ECG — Feature extraction (pair-level, after split)
# ===============================

import scipy.signal as signal

# Optional deps (Kaggle GPU etc.)
try:
    import cupy as cp
except Exception:
    cp = None

try:
    from tqdm import tqdm
except Exception:
    tqdm = None

class DINAMOECGProcessor:
    def __init__(self, fs=250):
        self.fs = fs  # Sampling frequency (Hz)
    
    def load_data(self, file_path):
        """Loads ECG data from CSV, handling different column structures."""
        try:
            df = pd.read_csv(file_path)
            if 'ECG' in df.columns:
                return df['ECG'].values
            if 'EcgWaveform' in df.columns:
                return df['EcgWaveform'].values
            if df.shape[1] > 1:
                return df.iloc[:, 1].values
            return df.iloc[:, 0].values
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
            return None

    def apply_filter(self, data):
        """Applies Bandpass filter (0.5-50Hz) to remove noise."""
        nyquist = 0.5 * self.fs
        low = 0.5 / nyquist
        high = 50.0 / nyquist
        b, a = signal.butter(1, [low, high], btype='band')
        return signal.filtfilt(b, a, data)

    def detect_r_peaks(self, clean_signal):
        """Standard Pan-Tompkins style peak detection (simplified)."""
        diff_sig = np.diff(clean_signal)
        squared_sig = diff_sig ** 2
        window_size = int(0.12 * self.fs)
        if window_size < 1:
            window_size = 1
        integrated_sig = np.convolve(squared_sig, np.ones(window_size)/window_size, mode='same')
        peaks, _ = signal.find_peaks(
            integrated_sig,
            distance=int(self.fs * 0.4),
            height=float(np.mean(integrated_sig)) if len(integrated_sig) else None,
        )
        return peaks

    def calculate_metrics(self, rr_intervals):
        """Calculates time-domain HRV metrics (GPU if available, else CPU)."""
        if rr_intervals is None or len(rr_intervals) < 2:
            return None

        if cp is not None:
            try:
                rr_gpu = cp.array(rr_intervals)
                mean_rr = cp.mean(rr_gpu)
                std_rr = cp.std(rr_gpu)
                rmssd = cp.sqrt(cp.mean(cp.diff(rr_gpu) ** 2))
                features = {
                    'mean_rr': float(mean_rr),
                    'std_rr': float(std_rr),
                    'rmssd': float(rmssd),
                    'mean_hr': 60.0 / (float(mean_rr) / 1000.0),
                }
                del rr_gpu
                cp.get_default_memory_pool().free_all_blocks()
                return features
            except Exception:
                pass

        mean_rr = float(np.mean(rr_intervals))
        std_rr = float(np.std(rr_intervals))
        rmssd = float(np.sqrt(np.mean(np.diff(rr_intervals) ** 2)))
        mean_hr = 60.0 / (mean_rr / 1000.0) if mean_rr > 0 else 0.0
        return {'mean_rr': mean_rr, 'std_rr': std_rr, 'rmssd': rmssd, 'mean_hr': mean_hr}

    def process_file(self, file_path):
        sig = self.load_data(file_path)
        if sig is None or len(sig) < self.fs * 10:
            return None
        clean_sig = self.apply_filter(sig)
        peaks = self.detect_r_peaks(clean_sig)
        if peaks is None or len(peaks) < 3:
            return None
        rr_intervals = np.diff(peaks) / self.fs * 1000.0  # ms
        return self.calculate_metrics(rr_intervals)

def extract_ecg_features_from_pairs(pairs_df: pd.DataFrame, fs: int = 250) -> pd.DataFrame:
    processor = DINAMOECGProcessor(fs=fs)
    records = []
    it = pairs_df.itertuples(index=False)
    if tqdm is not None:
        it = tqdm(list(it), total=len(pairs_df), desc='ECG feature extraction')
    for row in it:
        metrics = processor.process_file(row.file_path_ecg)
        if not metrics:
            continue
        metrics.update({
            'pair_id': row.pair_id,
            'subject_id': row.subject_id,
            'session_id': row.session_id,
            'y': int(row.y_pair),
            'file_path_ecg': row.file_path_ecg,
        })
        records.append(metrics)
    df = pd.DataFrame(records)
    if len(df) == 0:
        return pd.DataFrame(columns=['pair_id','subject_id','session_id','y','file_path_ecg','mean_rr','std_rr','rmssd','mean_hr'])
    df = df.fillna(0)
    return df

df_ecg_train = extract_ecg_features_from_pairs(pairs_train)
df_ecg_test  = extract_ecg_features_from_pairs(pairs_test)

print('ECG train features:', df_ecg_train.shape)
print('ECG test features:', df_ecg_test.shape)
print('ECG train subjects:', df_ecg_train['subject_id'].nunique() if len(df_ecg_train) else 0)
print('ECG test subjects:', df_ecg_test['subject_id'].nunique() if len(df_ecg_test) else 0)
display(df_ecg_train.head())

ECG feature extraction: 100%|██████████| 26/26 [04:50<00:00, 11.16s/it]

ECG train features: (100, 9)
ECG test features: (26, 9)
ECG train subjects: 16
ECG test subjects: 4


,mean_rr,std_rr,rmssd,mean_hr,pair_id,subject_id,session_id,y,file_path_ecg
0,2584.877632,16629.493518,21053.615014,23.211931,healthy003_2014_10_01-10_14_30,003,2014_10_01-10_14_30,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
1,784.901039,898.680117,1073.781032,76.442758,healthy003_2014_10_02-08_37_05,003,2014_10_02-08_37_05,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
2,727.497641,570.560590,700.792627,82.474494,healthy003_2014_10_03-08_37_23,003,2014_10_03-08_37_23,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
3,2498.389407,11561.364328,15193.160900,24.015472,healthy003_2014_10_04-08_47_14,003,2014_10_04-08_47_14,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
4,1834.379919,9831.948806,12860.373440,32.708601,healthy004_2014_10_01-06_25_47,004,2014_10_01-06_25_47,0,/kaggle/input/d1namo-ecg-glucose-data/healthy_...


## Breathing features 


In [ ]:
df_resp_train = pairs_train[['file_path_breath', 'y_pair', 'subject_id', 'pair_id', 'session_id']]

df_resp_test = pairs_test[['file_path_breath', 'y_pair', 'subject_id', 'pair_id', 'session_id']]

# Fonctions d'extraction (identiques à respiration.ipynb)
def apply_bandpass_filter(signal_data, fs, lowcut=0.05, highcut=0.7):
    nyquist = fs / 2
    low, high = lowcut / nyquist, highcut / nyquist
    b, a = sp_signal.butter(4, [low, high], btype='band')
    return sp_signal.filtfilt(b, a, signal_data)

def detect_breathing_cycles(signal_data, fs):
    distance = int(0.5 * fs)
    peaks, _ = find_peaks(signal_data, prominence=0.02, distance=distance)
    troughs, _ = find_peaks(-signal_data, prominence=0.02, distance=distance)
    return np.sort(peaks), np.sort(troughs)

def extract_features_from_signal(signal_data, fs=18):
    signal_data = np.array(signal_data).flatten()
    filtered = apply_bandpass_filter(signal_data, fs)
    peaks, troughs = detect_breathing_cycles(filtered, fs)

    if len(peaks) < 2:
        return None

    bbi = np.diff(peaks) / fs
    br = 60 / bbi

    br_mean = np.mean(br)
    br_std = np.std(br)
    rmssd = np.sqrt(np.mean(np.diff(bbi) ** 2))
    sdnn = np.std(bbi)
    cv = sdnn / np.mean(bbi) if np.mean(bbi) > 0 else 0

    ti_list, te_list, ie_ratios = [], [], []

    for i in range(len(peaks) - 1):
        ti = None
        if i < len(troughs):
            ti = (peaks[i] - troughs[i]) / fs if troughs[i] < peaks[i] else None
            if ti is not None and ti > 0:
                ti_list.append(ti)

        if i < len(troughs):
            for trough in troughs:
                if trough > peaks[i]:
                    te = (trough - peaks[i]) / fs
                    te_list.append(te)
                    if ti is not None and ti > 0:
                        ie_ratios.append(ti / te)
                    break

    ie_ratios = [x for x in ie_ratios if x is not None]

    ti_mean = np.mean(ti_list) if ti_list else 0
    ti_std = np.std(ti_list) if ti_list else 0
    te_mean = np.mean(te_list) if te_list else 0
    te_std = np.std(te_list) if te_list else 0
    ie_ratio = np.mean(ie_ratios) if ie_ratios else 0

    amplitudes = []
    for i in range(len(peaks)):
        if i < len(troughs):
            for j in range(len(troughs) - 1, -1, -1):
                if troughs[j] < peaks[i]:
                    amp = filtered[peaks[i]] - filtered[troughs[j]]
                    amplitudes.append(amp)
                    break

    amp_mean = np.mean(amplitudes) if amplitudes else 0
    amp_std = np.std(amplitudes) if amplitudes else 0

    freqs, psd = welch(filtered, fs, nperseg=min(256, len(filtered)))
    dominant_freq = freqs[np.argmax(psd)]

    power_low = np.sum(psd[(freqs >= 0.05) & (freqs < 0.2)])
    power_med = np.sum(psd[(freqs >= 0.2) & (freqs <= 0.5)])
    power_high = np.sum(psd[(freqs > 0.5) & (freqs <= 0.7)])

    psd_norm = psd / np.sum(psd)
    spectral_entropy = -np.sum(psd_norm * np.log2(psd_norm + 1e-12))
    spectral_centroid = np.sum(freqs * psd) / np.sum(psd)

    return {
        'BR_mean': br_mean,
        'BR_std': br_std,
        'RMSSD': rmssd,
        'SDNN': sdnn,
        'CV': cv,
        'Ti_mean': ti_mean,
        'Ti_std': ti_std,
        'Te_mean': te_mean,
        'Te_std': te_std,
        'IE_ratio': ie_ratio,
        'Amp_mean': amp_mean,
        'Amp_std': amp_std,
        'Dominant_Freq': dominant_freq,
        'Power_Low_Freq': power_low,
        'Power_Med_Freq': power_med,
        'Power_High_Freq': power_high,
        'Spectral_Entropy': spectral_entropy,
        'Spectral_Centroid': spectral_centroid,
    }


def extract_features_for_sessions(sessions_df, name: str):
    print(f"\nExtraction des features ({name})...")
    feature_list = []
    n_total = len(sessions_df)

    for idx, row in sessions_df.iterrows():
        try:
            df = pd.read_csv(row['file_path_breath'])
            b_signal = df["BreathingWaveform"].values

            features = extract_features_from_signal(b_signal)
            if features:
                features['y_pair'] = row['y_pair']
                features['subject_id'] = row['subject_id']
                features["pair_id"] = row["pair_id"]
                features["session_id"] = row["session_id"]
                features["file_path_breath"] = row["file_path_breath"]
                feature_list.append(features)

            if (idx + 1) % 10 == 0:
                print(f"  {idx + 1}/{n_total} sessions traitées")
        except Exception as e:
            print(f"  Erreur {row['file_path_breath']}: {e}")

    out = pd.DataFrame(feature_list)
    print(f"✓ {name}: features extraites: {out.shape}")
    return out


features_train_df = extract_features_for_sessions(df_resp_train, "TRAIN")
features_test_df = extract_features_for_sessions(df_resp_test, "TEST")

print("\nColonnes features (doivent être identiques):")
print("TRAIN:", list(features_train_df.columns))
print("TEST :", list(features_test_df.columns))

print("\nAperçu TRAIN features:")
display(features_train_df.head())
print("\nAperçu TEST features:")
display(features_test_df.head())

# IMPORTANT: pour la suite (training + fusion), on travaille sur les tables de features.
# On garde y_pair, mais on ajoute un alias 'y' pour être compatible avec les cellules ECG/fusion.
df_resp_train = features_train_df.copy()
df_resp_test = features_test_df.copy()

if 'y_pair' in df_resp_train.columns and 'y' not in df_resp_train.columns:
    df_resp_train['y'] = df_resp_train['y_pair'].astype(int)
    df_resp_test['y'] = df_resp_test['y_pair'].astype(int)

print("\nResp features ready:")
print("df_resp_train:", df_resp_train.shape, "| df_resp_test:", df_resp_test.shape)
display(df_resp_train[['pair_id','subject_id','session_id','y','y_pair']].head())


Extraction des features (TRAIN)...
  10/100 sessions traitées
  20/100 sessions traitées
  30/100 sessions traitées
  40/100 sessions traitées
  50/100 sessions traitées
  60/100 sessions traitées
  70/100 sessions traitées
  80/100 sessions traitées
  100/100 sessions traitées
  110/100 sessions traitées
  120/100 sessions traitées
✓ TRAIN: features extraites: (100, 23)

Extraction des features (TEST)...
  90/26 sessions traitées
✓ TEST: features extraites: (26, 23)

Colonnes features (doivent être identiques):
TRAIN: ['BR_mean', 'BR_std', 'RMSSD', 'SDNN', 'CV', 'Ti_mean', 'Ti_std', 'Te_mean', 'Te_std', 'IE_ratio', 'Amp_mean', 'Amp_std', 'Dominant_Freq', 'Power_Low_Freq', 'Power_Med_Freq', 'Power_High_Freq', 'Spectral_Entropy', 'Spectral_Centroid', 'y_pair', 'subject_id', 'pair_id', 'session_id', 'file_path_breath']
TEST : ['BR_mean', 'BR_std', 'RMSSD', 'SDNN', 'CV', 'Ti_mean', 'Ti_std', 'Te_mean', 'Te_std', 'IE_ratio', 'Amp_mean', 'Amp_std', 'Dominant_Freq', 'Power_Low_Freq', 'Power

,BR_mean,BR_std,RMSSD,SDNN,CV,Ti_mean,Ti_std,Te_mean,Te_std,IE_ratio,...,Power_Low_Freq,Power_Med_Freq,Power_High_Freq,Spectral_Entropy,Spectral_Centroid,y_pair,subject_id,pair_id,session_id,file_path_breath
0,25.067511,11.316291,1.394342,1.256548,0.434510,1.595152,0.780929,1.556676,0.894630,1.008271,...,1.592492e+10,9.233456e+09,1.186934e+09,2.820176,0.195614,0,003,healthy003_2014_10_01-10_14_30,2014_10_01-10_14_30,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
1,24.371905,11.125342,1.445125,1.304521,0.437242,1.296407,0.713998,1.611539,0.929990,1.187500,...,1.701656e+10,1.117802e+10,1.263880e+09,2.838536,0.197151,0,003,healthy003_2014_10_02-08_37_05,2014_10_02-08_37_05,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
2,24.533442,11.231087,1.503901,1.304974,0.440239,1.145378,0.569228,1.598424,0.922760,1.096349,...,6.692833e+09,2.902858e+09,1.862274e+08,2.549348,0.164259,0,003,healthy003_2014_10_03-08_37_23,2014_10_03-08_37_23,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
3,24.595035,10.751491,1.573127,1.267305,0.435070,0.000000,0.000000,1.444678,0.861069,0.000000,...,3.050973e+08,2.738831e+08,1.084719e+07,2.691483,0.186183,0,003,healthy003_2014_10_04-08_47_14,2014_10_04-08_47_14,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
4,24.619510,9.545791,1.270074,1.065483,0.380170,1.135638,0.596602,1.566757,0.834658,1.011036,...,1.591737e+10,8.535005e+09,1.023020e+09,2.664033,0.193738,0,004,healthy004_2014_10_01-06_25_47,2014_10_01-06_25_47,/kaggle/input/d1namo-ecg-glucose-data/healthy_...



Aperçu TEST features:


,BR_mean,BR_std,RMSSD,SDNN,CV,Ti_mean,Ti_std,Te_mean,Te_std,IE_ratio,...,Power_Low_Freq,Power_Med_Freq,Power_High_Freq,Spectral_Entropy,Spectral_Centroid,y_pair,subject_id,pair_id,session_id,file_path_breath
0,26.428033,11.184020,1.331778,1.154409,0.427792,1.274604,0.650804,1.440510,0.805569,1.129856,...,7.800278e+08,5.767696e+08,5.896574e+07,2.865520,0.196415,0,001,healthy001_2014_10_01-12_50_01,2014_10_01-12_50_01,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
1,26.773143,11.223433,1.405714,1.165125,0.437801,1.145062,0.620558,1.431819,0.848271,1.108216,...,5.938466e+08,5.269701e+08,4.540989e+07,2.897100,0.206713,0,001,healthy001_2014_10_02-07_11_17,2014_10_02-07_11_17,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
2,26.811026,10.770283,1.325226,1.142410,0.433027,5.748024,3.085722,1.422415,0.838526,5.611762,...,3.338155e+08,2.265618e+08,2.656428e+07,2.824381,0.202192,0,001,healthy001_2014_10_03-07_50_09,2014_10_03-07_50_09,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
3,28.634132,11.861409,1.365909,1.215801,0.480765,8.645888,4.381946,1.354328,0.875355,8.546112,...,1.110476e+09,7.851511e+08,8.956829e+07,2.866134,0.198766,0,001,healthy001_2014_10_04-07_31_03,2014_10_04-07_31_03,/kaggle/input/d1namo-ecg-glucose-data/healthy_...
4,25.849453,11.511375,1.520417,1.248739,0.445893,3.140392,1.694652,1.464273,0.865820,3.260955,...,9.498507e+08,5.439279e+08,4.197947e+07,2.676848,0.174970,0,002,healthy002_2014_10_01-06_17_49,2014_10_01-06_17_49,/kaggle/input/d1namo-ecg-glucose-data/healthy_...


In [15]:
# ===============================
# Export CSV — tables de features extraites 
# ===============================

from pathlib import Path

out_dir = Path.cwd() / "export_features"
out_dir.mkdir(parents=True, exist_ok=True)


def _save_df(df, name: str) -> None:
    path = out_dir / f"{name}.csv"
    df.to_csv(path, index=False)
    try:
        rows, cols = df.shape
    except Exception:
        rows, cols = "?", "?"
    print(f"Wrote: {path} | rows={rows} cols={cols}")


for _var in ["df_ecg_train", "df_ecg_test", "df_resp_train", "df_resp_test"]:
    if _var not in globals():
        print(f"Skip {_var}: not found (run feature extraction cells first)")
        continue
    _df = globals()[_var]
    if _df is None:
        print(f"Skip {_var}: is None")
        continue
    if hasattr(_df, "empty") and _df.empty:
        print(f"Skip {_var}: empty")
        continue
    _save_df(_df, _var)

print(f"Export folder: {out_dir.resolve()}")


Wrote: /kaggle/working/export_features/df_ecg_train.csv | rows=100 cols=9
Wrote: /kaggle/working/export_features/df_ecg_test.csv | rows=26 cols=10
Wrote: /kaggle/working/export_features/df_resp_train.csv | rows=100 cols=5
Wrote: /kaggle/working/export_features/df_resp_test.csv | rows=26 cols=5
Export folder: /kaggle/working/export_features
